### Tables of units and blades' map.

In [1]:
AMPSUB = {
    0    : 1.0,    # no unit defined.
    "mA" : 1e-3,   # mili
    "uA" : 1e-6,   # micro
    "nA" : 1e-9,   # nano
    "pA" : 1e-12,  # pico
    "fA" : 1e-15,  # femto
    "aA" : 1e-18,  # atto
}

# The XBPM beamlines.
BEAMLINENAME = {
    "CAT": "Cateretê",
    "CNB": "Carnaúba",
    "MGN": "Mogno",
    "MNC": "Manacá",
}

BLADEMAP = {
    "MNC": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    "MNC1": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    "MNC2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "CAT":  {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    # "CAT1": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},

    # ## To be checked: ## #
    # "CAT2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "CNB": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    # "CNB1": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    # "CNB2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "MGN": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    "MGN1": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "MGN2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "SIM":  {"TO": 'A', "TI": 'B', "BI": 'C', "BO": 'D'},
}

FILE_EXTENSION = ".pickle"    # Data file type.

### Procedures for 2025-06-11

#### Set of functions to read data from files and restructure them for each beamline, in columns for each blade's value, the undulator gap and the SR current.

In [2]:
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle  # noqa: S403


In [8]:
# Functions for opening, rearranging and plotting data.

def data_from_file(wdir):
    """Read data from pickle files."""
    allfiles = os.listdir(wdir)
    picklefiles = [pf for pf in allfiles if pf.endswith("pickle")]
    sfiles = sorted(picklefiles, key=lambda name: lastfield(name, '_'))

    rawdata = list()
    for file in sfiles:
        with open(wdir + "/" + file, 'rb') as df:
            rawdata.append(pickle.load(df))  # noqa: S301

    return rawdata


def lastfield(name, fld=' '):  # noqa: D103
    return name.split(fld)[-1]


def read_rawdata(rawdata):
    """."""
    data = dict()
    blk = 0
    for nline in range(6):
        beamline = rawdata[blk][0][nline]['name']
        data[beamline] = dict()

    for blk in range(len(rawdata)):
        for nline, bline in enumerate(data.keys()):
            data[bline][blk] = dict()
            # For each block.
            data[bline][blk]['data'] = deepcopy(rawdata[blk][0][nline])
            data[bline][blk]['info'] = deepcopy(rawdata[blk][1])

    return data


def data_structure_average(rdata):
    """Average over blades' values and simplify data structure."""
    data = dict()
    for bline in rdata.keys():
        for blk in range(len(rdata[bline].keys())):
            for bld in ["A", "B", "C", "D"]:
                blade = f"{bld}_val"
                av = average_blade(rdata[bline][blk]['data'][blade])
                rdata[bline][blk]['data'][blade] = av
                del rdata[bline][blk]['data'][f"{bld}_range"]
            del rdata[bline][blk]['data']["name"]
            del rdata[bline][blk]['data']["prefix"]

            rdata[bline][blk]['data']["current"] = \
                np.round(rdata[bline][blk]['info']["current"], decimals=-1)
            del rdata[bline][blk]['info']["current"]

            bl = bline[:3].lower()   # Gap info key.
            if bl in rdata[bline][blk]['info'].keys():
                rdata[bline][blk]['data']['gap'] = \
                    np.round(rdata[bline][blk]['info'][bl], decimals=0)

            del rdata[bline][blk]["info"]
            rdata[bline][blk] = rdata[bline][blk]["data"]

        dt = dict()
        for key, rd in rdata[bline].items():
            dt[key] = rd
        data[bline] = dt
    return data


def average_blade(blade):
    """Function for averaging over each blade's measurement."""
    bld = []
    for val in blade:
        bld.append(val[0] * AMPSUB[val[1]])
    return np.array([np.average(bld), np.std(bld)])


def pandas_data_frame(data):
    """Data to pandas' data frame."""
    pdata = dict()
    for key, val in data.items():
        pdata[key] = pd.DataFrame(val)
    return pdata


def blades_data_array(pdata, beamline):
    """Divide data into arrays for each blade."""
    blades = {
        "A_val" : [],
        "B_val" : [],
        "C_val" : [],
        "D_val" : [],
    }

    for ip, p in enumerate(pdata[beamline]):
        crr = np.round(pdata[beamline][p]["current"], decimals=-1)

        try:
            if beamline in ["CAT", "CNB"]:
                gap = np.round(pdata[beamline][p]["gap"], decimals=0)
            elif beamline in ["MNC1", "MNC2"]:
                gap = pdata[beamline][p]["gap"]
            else:
                gap = None  # pdata[beamline][p]["gap"]
        except Exception as err:
            print(f" WARNING: beamline {beamline}, data # {ip}:"
                  f"\t Exception when trying to define {err}")
            gap = None

        # print(f">>> (BLADES DATA RRAY) gap {beamline} = {gap}")

        for bl in blades.keys():
            blval = pdata[beamline][p][bl]
            # print(f" {cr} ({type(cr)}) : {bl} → {blval}")
            blades[bl].append([crr, gap, blval[0], blval[1]])

    for key, val in blades.items():
        blades[key] = np.array(val)

    return blades


def blades_plot(blades, ax, beamline):
    """Plot data."""
    # Plot markers for each blade.
    markers = {'A_val' : 'o',
               'B_val' : '^',
               'C_val' : 's',
               'D_val' : '*'}

    # Beamlines have different amperimeters.
    if beamline in ["CAT", "CNB"]:
        # Current in nA.
        y_unit = 1e9
        ylabel = u"$I$ [nA]"
        title_unit = "currents [nA]"
    else:
        # Current in counts.
        y_unit = 1.0
        ylabel = "counts"
        title_unit = "measures [counts]"

    for key, val in blades.items():

        x = val[:, 0]
        y = val[:, 2] * y_unit
        s = val[:, 3]

        # Define marker for each blade.
        mrk = markers[key]

        # Neglect data with spurious values.
        # mask = val[:, 2] >= -0.1
        mask = y >= -0.1

        # Plots for beamlines with undulator and dipole sources.
        if beamline in ["CAT", "CNB", "MNC1", "MNC2"]:
            # Beamlines with undulator source.
            gaps = np.unique(val[:, 1])

            # Divide in groups by undulator's gap.
            for gap in gaps:
                mask_gap = val[:, 1] == gap
                gmask = mask & mask_gap
                ax.errorbar(x[gmask], y[gmask], s[gmask], fmt=f'{mrk}-',
                            label=f"{key} @ gap {gap}")
        else:
            # Beamline with dipole source.
            ax.errorbar(x[mask], y[mask], s[mask],
                        fmt=f'{mrk}-', label=key)

    title = f"XBPM blades {title_unit} "
    if beamline in ["CAT", "CNB", "MNC1", "MNC2"]:
        title += u"$\\times$ undulator gaps @"
    else:
        title += u"$\\times$ SR currents @"
    ax.set_title(f"{title} {beamline}")
    ax.set_xlabel("SR current [mA]")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid()

In [4]:
%matplotlib qt5

## Main.

In [ ]:
def main(wdir, outdir):
    """."""
    # List with data read from working directory.
    rawdata = data_from_file(wdir)

    # Extract data from read files and compose a dict
    # relative to each beam line.
    rdata   = read_rawdata(rawdata)

    # Average over blades' measurements and
    # simplify data structure dictionary.
    data    = data_structure_average(rdata)

    # Export data to a pandas data frame.
    pdata   = pandas_data_frame(data)

    for beamline in pdata.keys():
        # Divide pdata into arrays for each blade.
        blades = blades_data_array(pdata, beamline=beamline)

        if beamline in ["CAT", "CNB", "MNC1", "MNC2"]:
            # Axes for current and gap (undulator) plots.
            fig, ax = plt.subplots(1, 1, figsize=(10, 6))
            blades_plot(blades, ax, beamline)
        else:
            # Axis for current plot only.
            fig, ax = plt.subplots(1, 1, figsize=(10, 6))
            blades_plot(blades, ax, beamline)

        fig.tight_layout()
        fig.savefig(f"{outdir}/XBPM_{beamline}_SR_current.png", dpi=300)

    plt.show()


# Working directory and output directory.
a = ""
basedir = a + "results_2025/"
outdir = basedir + "20250616_Gaps/"
wdir = outdir + "2025-06-16"

if __name__ == "__main__":
    main(wdir, outdir)

In [ ]:
wdir = ("results_2025/" + "20250616_Gaps/2025-06-16")

rawdata = data_from_file(wdir)
rdata   = read_rawdata(rawdata)
data    = data_structure_average(rdata)
pdata   = pandas_data_frame(data)

# for key, dt in pdata.items():
#     print(f"{key}: {dt}")

pdata['CNB'].T

In [ ]:
a = ""
wdir = a + "results_2025/20250623_Currents/"
datadir = wdir + "2025-06-23"

rawdata = data_from_file(datadir)

rdata = {key : {} for key in rawdata[0][0].keys()}
for rd in rawdata:
    for key, val in rd[0].items():
        current = np.round(rd[1]["current"], decimals=-1)
        rdata[key][current] = {}
        rdata[key][current]["gap"] = round(rd[1]['cnb'])

        for blade in ["A_val", "B_val", "C_val", "D_val"]:
            rdata[key][current][blade] = average_blade(val[blade])

pdata = pandas_data_frame(rdata)

# fsize = (6, 4)

for beamline in list(pdata.keys()):
    fig, ax = plt.subplots(1, 1)
    currents = np.array(pdata[beamline].keys())
    for blade in ['A_val', 'B_val', 'C_val', 'D_val']:
        bl, sbl = list(), list()
        for val in pdata[beamline].T[blade]:
            bl.append(val[0])
            sbl.append(val[1])
        bl, sbl = np.array(bl), np.array(sbl)
        if beamline in ["CAT", "CNB"]:
            ylabel = u"$I$ [$\\mu$A]"
            bl *= 1e6   # in uA
            sbl *= 1e6   # in uA
        else:
            ylabel = "counts"
        ax.errorbar(currents, bl, sbl, fmt='o-', label=blade)
        ax.set_title(f" XBPM measurements @ {beamline}")
        ax.set_xlabel("SR current [mA]")
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(visible=True)
        fig.savefig(wdir + f"XBPM_SR_current_at_{beamline}.png", dpi=300)

plt.show()


### 2025-07-29 : CNB (subsec 06SB)

#### Failed measurement.

All blades' values of each blade are equal, hence only one point is defined.
No other positions can be calculated.


In [ ]:
wdir = ("results_2025/20250729_CNB/" + "subsec_06SB")

rawdata = data_from_file(wdir)
# rdata = read_rawdata(rawdata)

for ii, rd in enumerate(rawdata):
    print(f"({ii:3d}) {rd[0]['CNB']['A_val']}")
